# Narrative Trails sobre el subset Taliban

El mismo método del notebook de bpoil, aplicado al subset "NEWS-TLS Entities
(Taliban)" del corpus de Afganistán. Es una exploración comparativa y no parte
formal del proyecto, igual que `scripts/afg_explore.py`. En este corpus los
títulos son titulares reales, así que las narrativas se imprimen sin el
comienzo del texto.

El método tiene tres pasos:

1. UMAP proyecta los embeddings (mpnet, 768 dimensiones) a 48 dimensiones y
   HDBSCAN agrupa esa proyección en tópicos. Cada documento queda con una
   distribución de pertenencia a los tópicos.
2. La coherencia entre dos documentos es la media geométrica entre la similitud
   angular de sus embeddings y la similitud de sus distribuciones de tópicos
   (Jensen-Shannon). Se descartan las aristas más débiles que la arista más
   débil del árbol de expansión máxima (la coherencia crítica), lo que deja el
   grafo conexo. Con la restricción de fechas, cada documento solo apunta a
   documentos del mismo día o posteriores.
3. Una narrativa entre un origen y un destino es el camino de capacidad máxima:
   el que maximiza la coherencia de su eslabón más débil (el bottleneck).
   `reliability` es la media geométrica de las coherencias del camino.

Paper y repositorio en las [referencias del README](../README.md#narrative-trails).

In [1]:
import warnings

# Dos avisos que no afectan el resultado: tqdm pide ipywidgets al importarse
# desde umap, y hdbscan 0.8.40 llama a scikit-learn 1.6 con un argumento que
# se renombró ("force_all_finite").
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message=".*force_all_finite.*")

import pandas as pd

from causal_coherence.data_loading import AFGHANISTAN_DIR, load_subset
from causal_coherence.narrative import (
    alternatives_summary,
    count_topics,
    extract_alternatives,
    load_or_build_landscape,
    node_degree_report,
    print_alternatives,
    random_pair_paths,
)

SOURCE_LABEL = "NEWS-TLS Entities (Taliban)"

df, embeddings = load_subset(SOURCE_LABEL, AFGHANISTAN_DIR)
landscape, coherence_hash = load_or_build_landscape(embeddings, df["date"])

print(f"{len(df)} documentos, de {df['date'].min().date()} a {df['date'].max().date()}")
print(f"tópicos (HDBSCAN): {count_topics(landscape.cluster_labels)}")
print(f"aristas en el grafo: {landscape.nx_graph.number_of_edges():,}")
print(f"hash de la matriz de coherencia: {coherence_hash}")

Landscape cargado desde caché: landscape_1c82b2909480e6bc.pkl
2670 documentos, de 2006-07-31 a 2017-09-19
tópicos (HDBSCAN): 134
aristas en el grafo: 1,981,546
hash de la matriz de coherencia: 8c4f0fd4ec69896e


## Caché del landscape

UMAP, HDBSCAN y el grafo de coherencia tardan unos 2 minutos en este corpus.
`load_or_build_landscape` guarda el landscape ajustado en `data/cache/` (git lo
ignora) y en las corridas siguientes lo carga en menos de un segundo. El nombre
del archivo es un hash de todo lo que determina el resultado: embeddings,
fechas, `LANDSCAPE_PARAMS` y las versiones de las librerías que lo cambian (ver
`docs/reproducibilidad.md`). Si cambia cualquiera de esas cosas, cambia el hash
y el landscape se vuelve a ajustar.

Junto al landscape se guarda el hash de la matriz de coherencia, el mismo
identificador con que `scripts/baseline_narrative.py` nombra sus resultados.

## Narrativas entre 15 y 727

Es el mismo par de `scripts/afg_explore.py`. Se eligió a mano para esta
exploración y no con un criterio documentado, como sí se hizo en bpoil.

In [2]:
SRC_NODE, TGT_NODE = 15, 727
N_PATHS = 3

storylines = extract_alternatives(landscape, SRC_NODE, TGT_NODE, N_PATHS)
display(alternatives_summary(storylines).round(3))
print_alternatives(df, landscape, storylines)

,largo,bottleneck,reliability
alternativa,,,
0,6,0.762,0.811
1,4,0.762,0.788
2,4,0.754,0.772


--- Alternativa 0: 6 documentos · bottleneck 0.762 · reliability 0.811
[15] 2006-08-27 · tópico 94
    British soldier shot dead in Afghanistan
[264] 2007-07-01 · tópico 90 · coherencia 0.862
    Two more British soldiers killed in clashes with the Taliban
[330] 2007-08-30 · tópico 90 · coherencia 0.867
    Taliban leader and RAF gunner killed in Afghanistan
[626] 2008-07-16 · tópico 59 · coherencia 0.778
    Afghanistan: Special forces kill Taliban leader in 'critical blow' to insurgency
[712] 2008-10-05 · tópico 127 · coherencia 0.762
    War on Taliban can't be won, says army chief
[727] 2008-10-15 · tópico 73 · coherencia 0.792
    Civilian dead are a trade-off in Nato's war of barbarity

--- Alternativa 1: 4 documentos · bottleneck 0.762 · reliability 0.788
[15] 2006-08-27 · tópico 94
    British soldier shot dead in Afghanistan
[21] 2006-09-03 · tópico 77 · coherencia 0.762
    Renewed offensive and soldiers' deaths show mission is far from accomplished
[20] 2006-09-03 · tópico 9

## Diagnóstico de grado

Una narrativa puede salir rara (muy corta o con un eslabón débil) porque uno de
sus extremos es un nodo periférico: después de podar las aristas bajo la
coherencia crítica y aplicar la restricción de fechas, le quedan pocas aristas y
el camino tiene pocas opciones para entrar o salir de él.

`node_degree_report` compara el grado de un nodo (aristas entrantes más
salientes) con el promedio del grafo. `percentile` es la fracción de nodos con
grado estrictamente menor, y `peripheral` marca los nodos con menos de la mitad
del grado promedio, el mismo criterio de `scripts/afg_explore.py`. La tabla
aplica el diagnóstico a cada documento de la primera narrativa.

In [3]:
chain = storylines[0].chain
roles = ["origen"] + ["intermedio"] * (len(chain) - 2) + ["destino"]
degrees = pd.DataFrame([node_degree_report(landscape, n) for n in chain]).set_index("node")
degrees.insert(0, "rol", roles)
degrees.round({"mean_degree": 1, "percentile": 3})

,rol,degree,mean_degree,percentile,peripheral
node,,,,,
15,origen,1997,1484.3,0.866,False
264,intermedio,1997,1484.3,0.866,False
330,intermedio,2002,1484.3,0.943,False
626,intermedio,1988,1484.3,0.553,False
712,intermedio,1988,1484.3,0.553,False
727,destino,2000,1484.3,0.925,False


Ninguno de los seis documentos es periférico: todos tienen grado entre 1988 y
2002, por encima del promedio del grafo (1484), y quedan entre el percentil 55
y el 94. El largo de esta narrativa no se explica por un extremo mal conectado.

## Pares al azar: qué largo de camino es normal

El largo de una narrativa (cuántos documentos tiene) depende de los extremos y
de la forma del grafo. Para saber si un largo es raro hace falta una referencia.
Este experimento extrae la narrativa entre 20 pares de documentos elegidos al
azar y registra el largo del camino, su bottleneck y cuántos días separan a los
dos extremos. Con eso se ve qué largo es típico en este grafo y si crece con la
separación temporal. En cada par, el documento de índice menor es el origen:
como `df` está ordenado por fecha, así el origen nunca queda después del
destino. La semilla es fija para que la tabla se repita.

El experimento se hizo primero sobre bpoil, con un ciclo que estaba en
`scripts/baseline_narrative.py` y que salió del script al limpiar los prints de
depuración. Ahora está en `random_pair_paths`.

In [4]:
pairs = random_pair_paths(landscape, df["date"], n_pairs=20, seed=0)
pairs.round(3)

,src,tgt,días,largo,bottleneck
0,1700,2270,1148,8,0.752
1,720,821,89,8,0.736
2,44,200,186,5,0.712
3,1733,2170,899,9,0.662
4,1344,1619,289,6,0.731
5,1688,1947,493,6,0.767
6,1494,2496,1940,7,0.771
7,1791,2177,860,9,0.738
8,1051,2289,1816,8,0.717
9,89,2042,2230,15,0.806


In [5]:
found = pairs.dropna(subset=["largo"])
print(f"pares con camino: {len(found)} de {len(pairs)}")
print(f"largo: mediana {found['largo'].median():.0f}, media {found['largo'].mean():.1f}, "
      f"mínimo {found['largo'].min():.0f}, máximo {found['largo'].max():.0f}")
print(f"correlación de Spearman entre días de separación y largo: {found['días'].corr(found['largo'], method='spearman'):.2f}")
print(f"largo de la primera narrativa 15 -> 727: {len(storylines[0].chain)}")

pares con camino: 20 de 20
largo: mediana 7, media 7.7, mínimo 3, máximo 16
correlación de Spearman entre días de separación y largo: 0.50
largo de la primera narrativa 15 -> 727: 6


## Qué se encontró

Los 20 pares tienen camino. El largo típico es de 7 documentos (mediana 7,
media 7,7), con casos entre 3 y 16. El largo crece con la separación temporal
(Spearman 0,50): los tres caminos más largos, de 12 a 16 documentos, unen
extremos separados por más de 2200 días. La narrativa 15 → 727 tiene 6
documentos para 780 días de separación y queda dentro de lo normal para este
grafo, así que su largo no necesita una explicación particular.

16 de los 20 bottleneck están entre 0,65 y 0,82. Los más bajos son
1403 → 1789 (0,45), 686 → 1643 (0,44) y 1829 → 2536 (0,57). Son los pares
que conviene revisar para estudiar en qué casos el método no encuentra un hilo
coherente.